# s2figure.ipynb

Manuscript analysis and plotting notebook.

Run the configuration cell first. Input data and fonts are not bundled.
See the repository README and data requirements before executing.


In [ ]:
from pathlib import Path
import sys as _sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'plot_config.py').is_file()
     and (p / 'config.example.json').is_file()),
    None,
)
if _repo_root is None:
    raise RuntimeError('Run this notebook from within the plotting repository.')
if str(_repo_root) not in _sys.path:
    _sys.path.insert(0, str(_repo_root))
from plot_config import configured_path


# Locations of the four ocean stations

Robinson-projection overview of BAT, TAO, HOT, and Papa. The displayed longitude range is 30°E–390°E, centered on 210°E.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.transforms import offset_copy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# Locate the plot directory whether this notebook is run from the repository
# root or directly from plot1.
plot_dir = Path(configured_path('work_dir'))
font_dir = Path(configured_path('font_dir'))
font_files = [
    font_dir / 'Times New Roman.ttf',
    font_dir / 'Times New Roman Bold.ttf',
]
for font_file in font_files:
    if not font_file.exists():
        raise FileNotFoundError(f'Required font not found: {font_file}')
    font_manager.fontManager.addfont(font_file)
resolved_times = Path(
    font_manager.findfont('Times New Roman', fallback_to_default=False)
).resolve()
if resolved_times not in {font.resolve() for font in font_files}:
    raise RuntimeError(f'Times New Roman resolved unexpectedly: {resolved_times}')

# Apply Times New Roman to every text element.
mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
})

stations = {
    'BAT':    {'lat': 31.67,  'lon': 295.83, 'offset': (10, 0),  'ha': 'left'},
    'TAO':    {'lat': 8.00,   'lon': 137.00, 'offset': (10, 0),  'ha': 'left'},
    'HOT':    {'lat': 22.75,  'lon': 202.00, 'offset': (-10, 0), 'ha': 'right'},
    'Papa':   {'lat': 50.13,  'lon': 215.17, 'offset': (-10, 0), 'ha': 'right'},
    'SOTS':   {'lat': -47.00, 'lon': 142.00, 'offset': (10, 0),  'ha': 'left'},
    'PIRATA': {'lat': -10.00, 'lon': 350.00, 'offset': (-10, 0), 'ha': 'right'},
}

data_crs = ccrs.PlateCarree()
fig = plt.figure(figsize=(12, 6.2), dpi=150)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson(central_longitude=210))
ax.set_global()  # Robinson seam is at 30°E/390°E for central_longitude=210°.
ax.set_facecolor('white')
ax.add_feature(cfeature.LAND, facecolor='#d3d3d3', edgecolor='black', linewidth=0.8, zorder=1)
ax.coastlines(color='black', linewidth=0.8, zorder=2)
ax.spines['geo'].set_linewidth(1.1)
ax.spines['geo'].set_edgecolor('black')

# Sparse boundary coordinate labels, matching the reference style. The
# gridline width is zero so no longitude/latitude lines appear on the map.
gl = ax.gridlines(
    crs=data_crs, draw_labels=True, x_inline=False, y_inline=False,
    xlocs=[30, 150, -90], ylocs=[-45, 0, 45],
    linewidth=0, color='none', zorder=3,
)
gl.top_labels = False
gl.right_labels = False
gl.rotate_labels = False
gl.xformatter = LONGITUDE_FORMATTER
gl.yformatter = LATITUDE_FORMATTER
gl.xlabel_style = {'size': 24, 'family': 'Times New Roman'}
gl.ylabel_style = {'size': 24, 'family': 'Times New Roman'}
# Repeat 30°E at the right-hand 390°E seam, as in the reference figure.
ax.text(
    0.765, -0.012, '30°E', transform=ax.transAxes,
    ha='center', va='top', fontsize=24, family='Times New Roman',
    clip_on=False, zorder=6,
)

for name, station in stations.items():
    lon = ((station['lon'] - 30) % 360) + 30
    lat = station['lat']
    ax.scatter(
        lon, lat, s=115, color='red', edgecolor='black', linewidth=1.1,
        transform=data_crs, zorder=5,
    )
    name_transform = offset_copy(
        data_crs._as_mpl_transform(ax), fig=fig,
        x=station['offset'][0], y=station['offset'][1], units='points',
    )
    ax.text(
        lon, lat, name, transform=name_transform,
        ha=station['ha'], va='center', fontsize=26, fontweight='bold',
        color='black', zorder=6,
    )

fig.tight_layout(pad=0)
output_dir = Path(configured_path('output_dir'))
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / 's2figure.png'
fig.savefig(
    output_file, dpi=600, bbox_inches='tight', pad_inches=0,
    facecolor='white',
)
plt.show()
print(f'Saved figure to: {output_file.resolve()}')
